# PCA Analysis

This notebook runs PCA on role vectors and examines the top PCs.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
from pathlib import Path
from huggingface_hub import snapshot_download
from huggingface_hub.utils import disable_progress_bars

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from assistant_axis import compute_pca, MeanScaler, load_role_vector, slot_labels

disable_progress_bars()

## Configuration

In [ ]:
# Model configuration
MODEL_NAME = "qwen-3-32b"
TARGET_LAYER = 24

# Local paths if you computed the vectors yourself
LOCAL_ROLE_VECTORS_DIR = Path(f"../../outputs/{MODEL_NAME}/roles/vectors")
LOCAL_DEFAULT_VECTOR_PATH = Path(f"../../outputs/{MODEL_NAME}/roles/vectors/default.pt")

# HuggingFace configuration for pre-computed vectors
# Models supported: gemma-2-27b, qwen-3-32b, llama-3.3-70b
REPO_ID = "lu-christina/assistant-axis-vectors"

## Load Data

Run **one** of the two sections below.

In [ ]:
# # HuggingFace
# print(f"Loading from HuggingFace: {REPO_ID}")
# 
# # Download all vectors for this model
# local_dir = snapshot_download(
#     repo_id=REPO_ID,
#     repo_type="dataset",
#     allow_patterns=[f"{MODEL_NAME}/role_vectors/*.pt", f"{MODEL_NAME}/default_vector.pt"]
# )
# 
# # Load role vectors (handles both bare-tensor and dict-wrapped formats)
# first_meta = {}
# role_vectors = {}
# for p in sorted(Path(local_dir, MODEL_NAME, "role_vectors").glob("*.pt")):
#     vec, meta = load_role_vector(p)
#     if vec.ndim == 2:
#         vec = vec.unsqueeze(0)
#     role_vectors[p.stem] = vec
#     if not first_meta and meta:
#         first_meta = meta
# print(f"Loaded {len(role_vectors)} role vectors")
# 
# # Load default vector
# default_vector, default_meta = load_role_vector(Path(local_dir, MODEL_NAME, "default_vector.pt"))
# if default_vector.ndim == 2:
#     default_vector = default_vector.unsqueeze(0)
# print(f"Default vector shape: {default_vector.shape}")
# 
# # Determine slot labels from metadata (or from default_meta)
# labels = slot_labels(first_meta or default_meta)
# num_slots = next(iter(role_vectors.values())).shape[0]
# print(f"Slots ({num_slots}): {labels}")

In [ ]:
# FOR LOCAL

print(f"Loading from local: {LOCAL_ROLE_VECTORS_DIR}")

first_meta = {}
role_vectors = {}
for p in sorted(LOCAL_ROLE_VECTORS_DIR.glob("*.pt")):
    print(f"{p}")
    vec, meta = load_role_vector(p)
    if vec.ndim == 2:
        vec = vec.unsqueeze(0)
    role_vectors[p.stem] = vec
    if not first_meta and meta:
        first_meta = meta
print(f"Loaded {len(role_vectors)} role vectors")

default_vector, default_meta = load_role_vector(LOCAL_DEFAULT_VECTOR_PATH)
if default_vector.ndim == 2:
    default_vector = default_vector.unsqueeze(0)
print(f"Default vector shape: {default_vector.shape}")

labels = slot_labels(first_meta or default_meta)
num_slots = next(iter(role_vectors.values())).shape[0]
print(f"Slots ({num_slots}): {labels}")

## Run PCA

In [ ]:
# Run PCA for each slot at TARGET_LAYER
role_labels = list(role_vectors.keys())

pca_results = []
for s in range(num_slots):
    role_vectors_at_layer = torch.stack([v[s, TARGET_LAYER] for v in role_vectors.values()]).float()

    scaler = MeanScaler()
    pca_transformed, variance_explained, n_components, pca, scaler = compute_pca(
        role_vectors_at_layer, layer=None, scaler=scaler
    )
    pca_results.append({
        "slot_idx": s,
        "label": labels[s],
        "vectors_at_layer": role_vectors_at_layer,
        "pca_transformed": pca_transformed,
        "variance_explained": variance_explained,
        "n_components": n_components,
        "pca": pca,
        "scaler": scaler,
    })
    print(f"Slot {s} ({labels[s]}): fitted PCA with {len(variance_explained)} components")

## Plot variance explained

In [ ]:
def plot_variance_explained(ax, variance_explained, title, max_components=60, show_ylabel=True):
    """Plot variance explained (histogram + cumulative line)."""
    n_show = min(len(variance_explained), max_components)
    var_exp = variance_explained[:n_show]
    cumulative = np.cumsum(var_exp)
    components = np.arange(1, n_show + 1)
    
    bar_color = '#6a9bc3'
    line_color = '#1a5276'
    
    ax.bar(components, var_exp * 100, width=0.8, color=bar_color, alpha=0.6, 
           edgecolor='none', label='Individual')
    ax.plot(components, cumulative * 100, color=line_color, linewidth=1, label='Cumulative')
    
    # Threshold lines at 70%, 80%, 90%
    for thresh in [70, 80, 90]:
        idx = np.argmax(cumulative >= thresh / 100.0)
        if cumulative[idx] >= thresh / 100.0:
            n_dims = idx + 1
            ax.axhline(y=thresh, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
            ax.text(max_components - 1, thresh + 1.5, f'{thresh}% ({n_dims}d)', 
                    fontsize=7, color='gray', ha='right', va='bottom')
    
    ax.set_xlim(0, max_components + 1)
    ax.set_ylim(0, 105)
    ax.set_xlabel('Principal Component')
    if show_ylabel:
        ax.set_ylabel('Variance Explained (%)')
    ax.set_title(title, fontsize=11)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xticks([0, 20, 40, 60])
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.legend(loc='center right', fontsize=8)

model_title = MODEL_NAME.replace('-', ' ').title()
for result in pca_results:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    plot_variance_explained(
        ax, result["variance_explained"],
        f"Variance Explained — {result['label']} ({model_title})",
        max_components=60,
    )
    plt.tight_layout()
    plt.show()

## Compare Scree plots

In [ ]:
SLOT_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

def display_label(label):
    """Escape newline for display in plot titles."""
    return label.replace('\n', '\\n')

def plot_variance_explained(ax, variance_explained, title, max_components=60, show_ylabel=True):
    """Plot variance explained (histogram + cumulative line)."""
    n_show = min(len(variance_explained), max_components)
    var_exp = variance_explained[:n_show]
    cumulative = np.cumsum(var_exp)
    components = np.arange(1, n_show + 1)
    
    bar_color = '#6a9bc3'
    line_color = '#1a5276'
    
    ax.bar(components, var_exp * 100, width=0.8, color=bar_color, alpha=0.6, 
           edgecolor='none', label='Individual')
    ax.plot(components, cumulative * 100, color=line_color, linewidth=1, label='Cumulative')
    
    for thresh in [70, 80, 90]:
        idx = np.argmax(cumulative >= thresh / 100.0)
        if cumulative[idx] >= thresh / 100.0:
            n_dims = idx + 1
            ax.axhline(y=thresh, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
            ax.text(max_components - 1, thresh + 1.5, f'{thresh}% ({n_dims}d)', 
                    fontsize=7, color='gray', ha='right', va='bottom')
    
    ax.set_xlim(0, max_components + 1)
    ax.set_ylim(0, 105)
    ax.set_xlabel('Principal Component')
    if show_ylabel:
        ax.set_ylabel('Variance Explained (%)')
    ax.set_title(title, fontsize=11)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xticks([0, 20, 40, 60])
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.legend(loc='center right', fontsize=8)


def plot_scree_log(ax, variance_explained, title, log_x=False, max_components=60):
    """Plot individual variance explained on log scale."""
    n_show = min(len(variance_explained), max_components)
    var_exp = variance_explained[:n_show]
    components = np.arange(1, n_show + 1)

    ax.plot(components, var_exp * 100, color='#1a5276', linewidth=1.2, marker='.', markersize=3)
    ax.set_yscale('log')
    if log_x:
        ax.set_xscale('log')
    ax.set_xlabel('Principal Component')
    ax.set_ylabel('Variance Explained (%)')
    ax.set_title(title, fontsize=11)
    ax.grid(True, which='both', alpha=0.3, linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


model_title = MODEL_NAME.replace('-', ' ').title()
max_components = 100

# Per-slot rows
for result in pca_results:
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
    slot_label = display_label(result['label'])
    ve = result["variance_explained"]

    plot_variance_explained(
        axes[0], ve,
        f"Linear — {slot_label} ({model_title})",
        max_components=max_components,
    )
    plot_scree_log(
        axes[1], ve,
        f"Log-linear — {slot_label} ({model_title})",
        log_x=False, max_components=max_components,
    )
    plot_scree_log(
        axes[2], ve,
        f"Log-log — {slot_label} ({model_title})",
        log_x=True, max_components=max_components,
    )
    plt.tight_layout()
    plt.show()

# All slots superimposed
fig, axes = plt.subplots(1, 3, figsize=(16, 4))


for i, result in enumerate(pca_results):
    ve = result["variance_explained"]
    n_show = min(len(ve), max_components)
    var_exp = ve[:n_show]
    components = np.arange(1, n_show + 1)
    color = SLOT_COLORS[i]
    lbl = display_label(result['label'])

    # Linear: interleaved bars + cumulative lines
    width = 0.8 / num_slots
    offset = (i - (num_slots - 1) / 2) * width
    axes[0].bar(components + offset, var_exp * 100, width=width, color=color, alpha=0.6,
                edgecolor='none')
    cumulative = np.cumsum(var_exp)
    axes[0].plot(components, cumulative * 100, color=color, linewidth=1.2, label=lbl)
    
    # Log-linear
    axes[1].plot(components, var_exp * 100, color=color, linewidth=1.2,
                 marker='.', markersize=3, label=lbl)

    # Log-log
    axes[2].plot(components, var_exp * 100, color=color, linewidth=1.2,
                 marker='.', markersize=3, label=lbl)

axes[0].set_xlim(0, max_components + 1)
axes[0].set_ylim(0, None)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title(f'Linear — All slots ({model_title})', fontsize=11)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].legend(fontsize=8)

for ax, title_prefix, log_x in [(axes[1], 'Log-linear', False), (axes[2], 'Log-log', True)]:
    ax.set_yscale('log')
    if log_x:
        ax.set_xscale('log')
    ax.set_xlabel('Principal Component')
    ax.set_ylabel('Variance Explained (%)')
    ax.set_title(f'{title_prefix} — All slots ({model_title})', fontsize=11)
    ax.grid(True, which='both', alpha=0.3, linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Principal (a.k.a. canonical) angles between full PCA subspaces across slots
from scipy.linalg import svdvals
import itertools

N_ANGLES_SHOW = 100
N_RANDOM_TRIALS = 100

PAIR_COLORS = {
    (0, 1): 'red',      # body-mean vs <|im_start|>
    (0, 2): 'blue',     # body-mean vs assistant
    (0, 3): 'green',    # body-mean vs \n
    (1, 2): 'orange',   # <|im_start|> vs assistant
    (1, 3): 'cyan',     # <|im_start|> vs \n
    (2, 3): 'purple',   # assistant vs \n
}

def display_label(lbl):
    return lbl.replace('\n', '\\n')

def plot_principal_angles(pca_results, n_show, n_random_trials, model_title, pair_colors):
    n_total = min(r["n_components"] for r in pca_results)
    n_show = min(n_show, n_total)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    all_cos = []
    for (s1, s2), color in pair_colors.items():
        lbl1 = display_label(pca_results[s1]["label"])
        lbl2 = display_label(pca_results[s2]["label"])
        label = f'{lbl1} vs {lbl2}'

        U1 = pca_results[s1]["pca"].components_
        U2 = pca_results[s2]["pca"].components_

        cos_angles = svdvals(U1 @ U2.T)[:n_show]
        all_cos.append(cos_angles)

        ks = np.arange(1, n_show + 1)
        axes[0].plot(ks, cos_angles, color=color, linewidth=1.2, alpha=0.8, label=label)
        axes[1].plot(ks, cos_angles, color=color, linewidth=1.2, alpha=0.8, label=label)

    rng = np.random.default_rng(42)
    D = pca_results[0]["pca"].components_.shape[1]
    n_bases = n_total
    all_trials = []
    for trial in range(n_random_trials):
        random_bases = []
        for _ in range(4):
            M = rng.standard_normal((n_bases, D))
            Q, _ = np.linalg.qr(M.T)
            random_bases.append(Q.T[:n_bases])

        for r1, r2 in itertools.combinations(range(4), 2):
            cos_rand = svdvals(random_bases[r1] @ random_bases[r2].T)[:n_show]
            all_trials.append(cos_rand)

    random_mean = np.mean(all_trials, axis=0)
    all_cos.append(random_mean)

    ks = np.arange(1, n_show + 1)
    axes[0].plot(ks, random_mean, color='gray', linewidth=1.5, linestyle=':', alpha=0.8, label='random baseline')
    axes[1].plot(ks, random_mean, color='gray', linewidth=1.5, linestyle=':', alpha=0.8, label='random baseline')

    axes[1].set_yscale('log')

    # Derive y-limits from data
    global_min = min(c[-1] for c in all_cos)
    lin_floor = max(0, np.floor(global_min * 10) / 10 - 0.05)
    log_floor = 10 ** np.floor(np.log10(max(global_min, 1e-6)))

    for ax in axes:
        for x in range(0, n_show + 1, 10):
            ax.axvline(x, color="gray", linewidth=0.5, alpha=0.4)
        for x in range(0, n_show + 1, 50):
            ax.axvline(x, color="gray", linewidth=0.8, alpha=0.6)
        ax.set_xlabel('Principal angle index', fontsize=13)
        ax.set_ylabel('cos(θ_i)', fontsize=13)
        ax.set_xlim(1, n_show)
        ax.tick_params(labelsize=11)
        ax.legend(fontsize=9, ncol=2)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    axes[0].set_ylim(lin_floor, 1.05)
    axes[0].set_title(f'Principal angles: full PCA subspaces ({model_title})', fontsize=14)
    axes[1].set_ylim(log_floor, 1.5)
    axes[1].set_title(f'Principal angles (log scale)', fontsize=14)

    plt.tight_layout()
    plt.show()

plot_principal_angles(pca_results, N_ANGLES_SHOW, N_RANDOM_TRIALS, model_title, PAIR_COLORS)

In [ ]:
# Principal (a.k.a. canonical) angles between full PCA subspaces across slots
from scipy.linalg import svdvals
from sklearn.decomposition import PCA as _PCA
import itertools

N_ANGLES_SHOW = 100
N_RANDOM_TRIALS = 25
SUBSPACE_DIMS = [150]  # truncations to plot

PAIR_COLORS = {
    (0, 1): 'red',      # body-mean vs <|im_start|>
    (0, 2): 'blue',     # body-mean vs assistant
    (0, 3): 'green',    # body-mean vs \n
    (1, 2): 'orange',   # <|im_start|> vs assistant
    (1, 3): 'cyan',     # <|im_start|> vs \n
    (2, 3): 'purple',   # assistant vs \n
}

def display_label(lbl):
    return lbl.replace('\n', '\\n')

def plot_principal_angles(pca_results, n_show, n_random_trials, subspace_dims, model_title, pair_colors):
    n_total = min(r["n_components"] for r in pca_results)
    subspace_dims = [d for d in subspace_dims if d <= n_total]
    n_show = min(n_show, n_total)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    all_cos_min = []

    for (s1, s2), color in pair_colors.items():
        lbl1 = display_label(pca_results[s1]["label"])
        lbl2 = display_label(pca_results[s2]["label"])

        U1_full = pca_results[s1]["pca"].components_
        U2_full = pca_results[s2]["pca"].components_

        for dim in subspace_dims:
            U1 = U1_full[:dim]
            U2 = U2_full[:dim]
            cos_angles = svdvals(U1 @ U2.T)
            n_plot = min(n_show, len(cos_angles))
            cos_plot = cos_angles[:n_plot]
            all_cos_min.append(cos_plot[-1])

            alpha = 0.4 + 0.5 * (dim / max(subspace_dims))
            lw = 0.8 if dim < n_total else 1.4
            label = f'{lbl1} vs {lbl2}' if dim == max(subspace_dims) else None

            ks = np.arange(1, n_plot + 1)
            axes[0].plot(ks, cos_plot, color=color, linewidth=lw, alpha=alpha, label=label)
            axes[1].plot(ks, cos_plot, color=color, linewidth=lw, alpha=alpha, label=label)

    # Random baseline: fit PCA on random data, then compute principal angles at each truncation
    rng = np.random.default_rng(42)
    D = pca_results[0]["pca"].components_.shape[1]
    N = n_total

    # Pre-fit all random PCAs
    random_pcas = []
    for trial in range(n_random_trials):
        trial_pcas = []
        for _ in range(4):
            X = rng.standard_normal((N, D)).astype(np.float32)
            X -= X.mean(axis=0)
            pca_rand = _PCA()
            pca_rand.fit(X)
            trial_pcas.append(pca_rand)
        random_pcas.append(trial_pcas)

    for dim in subspace_dims:
        all_trials = []
        for trial_pcas in random_pcas:
            for r1, r2 in itertools.combinations(range(4), 2):
                U1 = trial_pcas[r1].components_[:dim]
                U2 = trial_pcas[r2].components_[:dim]
                cos_rand = svdvals(U1 @ U2.T)
                n_plot = min(n_show, len(cos_rand))
                all_trials.append(cos_rand[:n_plot])

        random_mean = np.mean(all_trials, axis=0)
        all_cos_min.append(random_mean[-1])

        alpha = 0.4 + 0.5 * (dim / max(subspace_dims))
        lw = 1.0 if dim < n_total else 1.5
        label = 'random baseline' if dim == max(subspace_dims) else None

        ks = np.arange(1, len(random_mean) + 1)
        axes[0].plot(ks, random_mean, color='gray', linewidth=lw, linestyle=':', alpha=alpha, label=label)
        axes[1].plot(ks, random_mean, color='gray', linewidth=lw, linestyle=':', alpha=alpha, label=label)

    # Derive y-limits from data
    global_min = min(all_cos_min)
    lin_floor = max(0, np.floor(global_min * 10) / 10 - 0.05)
    log_floor = 10 ** np.floor(np.log10(max(global_min, 1e-6)))

    axes[1].set_yscale('log')

    for ax in axes:
        for x in range(0, n_show + 1, 10):
            ax.axvline(x, color="gray", linewidth=0.5, alpha=0.4)
        for x in range(0, n_show + 1, 50):
            ax.axvline(x, color="gray", linewidth=0.8, alpha=0.6)
        ax.set_xlabel('Principal angle index', fontsize=13)
        ax.set_ylabel('cos(θ_i)', fontsize=13)
        ax.set_xlim(1, n_show)
        ax.tick_params(labelsize=11)
        ax.legend(fontsize=9, ncol=2)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    axes[0].set_ylim(lin_floor, 1.05)
    axes[0].set_title(f'Principal angles: PCA subspaces ({model_title})', fontsize=14)
    axes[1].set_ylim(log_floor, 1.5)
    axes[1].set_title(f'Principal angles (log scale)', fontsize=14)

    plt.tight_layout()
    plt.show()

plot_principal_angles(pca_results, N_ANGLES_SHOW, N_RANDOM_TRIALS, SUBSPACE_DIMS, model_title, PAIR_COLORS)

## Cosine Similarity with Top N PCs

In [ ]:
def plot_pc_lines(role_cosine_sims, role_labels, default_cosine_sims=None,
                       figsize=(8, 7), n_extremes=5, show_histogram=True, n_pcs=3):
    """
    Plot top N PCs' cosine similarities in a single figure with N subplots.
    """
    custom_cmap = LinearSegmentedColormap.from_list('PurpleTeal', ['#9b59b6', '#1abc9c'])
    
    n_per_quadrant = (n_extremes + 1) // 2
    n_tiers = max(2, (n_per_quadrant + 1) // 2)
    tier_spacing = 0.14
    y_max = 0.25 + n_tiers * tier_spacing + 0.1
    y_tiers_above = [0.25 + i * tier_spacing for i in range(n_tiers)]
    y_tiers_below = [-(0.25 + i * tier_spacing) for i in range(n_tiers)]
    
    height_per_pc = figsize[1] / 3
    fig, axes = plt.subplots(n_pcs, 1, figsize=(figsize[0], height_per_pc * n_pcs + n_tiers * 0.3))
    if n_pcs == 1:
        axes = [axes]
    
    for pc_idx, ax in enumerate(axes):
        projections = role_cosine_sims[:, pc_idx]
        
        c_norm = (projections + 1) / 2
        colors = custom_cmap(c_norm)
        
        sorted_indices = np.argsort(projections)
        low_indices = sorted_indices[:n_extremes].tolist()
        high_indices = sorted_indices[-n_extremes:][::-1].tolist()
        
        y_pos = np.zeros_like(projections)
        ax.scatter(projections, y_pos, c=colors, marker='o', s=40, alpha=0.6, edgecolors='none', zorder=3)
        
        if show_histogram:
            hist_counts, bin_edges = np.histogram(projections, bins=30, density=True)
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
            bin_width = bin_edges[1] - bin_edges[0]
            scaled_heights = hist_counts * 0.4
            bin_norm = (bin_centers + 1) / 2
            bin_colors = custom_cmap(np.clip(bin_norm, 0, 1))
            ax.bar(bin_centers, scaled_heights, width=bin_width, alpha=0.3, color=bin_colors, edgecolor='none', zorder=1)
        
        if default_cosine_sims is not None:
            val = default_cosine_sims[pc_idx]
            ax.axvline(x=val, color='blue', linestyle='--', linewidth=1, alpha=0.9, zorder=2)
            ax.scatter([val], [0], c='blue', marker='*', s=300, alpha=1.0, zorder=5)
            ax.text(val, y_tiers_above[-1] + 0.12, 'Default Response', ha='center', va='bottom', fontsize=10, color='blue', alpha=0.9)
        
        def u_tier(rank, n):
            """U-shaped: edges at tier 0 (closest to axis), middle at highest tier."""
            mid = (n - 1) / 2
            return int(mid - abs(rank - mid))
        
        def place_labels(indices, is_right=False):
            above = indices[0::2]
            below = indices[1::2]
            
            for rank, idx in enumerate(above):
                label = role_labels[idx].replace('_', ' ').title()
                x_pos = projections[idx]
                tier = u_tier(rank, len(above))
                y_label = y_tiers_above[min(tier, len(y_tiers_above) - 1)]
                is_peak = (len(above) % 2 == 1) and (rank == len(above) // 2)
                if is_peak:
                    ha = 'center'
                elif is_right:
                    ha = 'left' if rank < len(above) / 2 else 'right'
                else:
                    ha = 'right' if rank < len(above) / 2 else 'left'
                ax.plot([x_pos, x_pos], [0.02, y_label - 0.02],
                        '-', color='gray', alpha=0.4, linewidth=0.8, zorder=1)
                ax.text(x_pos, y_label, label, ha=ha, va='bottom', fontsize=8, zorder=4)
            
            for rank, idx in enumerate(below):
                label = role_labels[idx].replace('_', ' ').title()
                x_pos = projections[idx]
                tier = u_tier(rank, len(below))
                y_label = y_tiers_below[min(tier, len(y_tiers_below) - 1)]
                is_peak = (len(below) % 2 == 1) and (rank == len(below) // 2)
                if is_peak:
                    ha = 'center'
                elif is_right:
                    ha = 'left' if rank < len(below) / 2 else 'right'
                else:
                    ha = 'right' if rank < len(below) / 2 else 'left'
                ax.plot([x_pos, x_pos], [-0.02, y_label + 0.02],
                        '-', color='gray', alpha=0.4, linewidth=0.8, zorder=1)
                ax.text(x_pos, y_label, label, ha=ha, va='top', fontsize=8, zorder=4)
        
        place_labels(low_indices, is_right=False)
        place_labels(high_indices, is_right=True)
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['bottom'].set_position('zero')
        ax.axhline(y=0, color='black', linestyle='-', linewidth=2, zorder=1)
        ax.tick_params(axis='x', length=12, width=1.5, pad=10)
        ax.tick_params(axis='y', length=0, width=0)
        ax.set_yticks([])
        ax.set_xticks([-0.8, -0.4, 0, 0.4, 0.8])
        ax.set_ylim(-y_max, y_max + 0.05)
        ax.set_xlim(-1, 1)
        ax.grid(False)
        ax.set_title(f'PC{pc_idx + 1}', fontsize=12, fontweight='bold', loc='left')
    
    plt.tight_layout()
    return fig

In [ ]:
N_PCS = 5  # change to 5, 10, etc.

model_title = MODEL_NAME.replace('-', ' ').title()
for result in pca_results:
    s = result["slot_idx"]
    pca_obj = result["pca"]
    scaler_obj = result["scaler"]
    vectors_at_layer = result["vectors_at_layer"]

    n_pcs = min(N_PCS, pca_obj.n_components_)
    pc_directions = pca_obj.components_[:n_pcs]
    pc_directions = pc_directions / np.linalg.norm(pc_directions, axis=1, keepdims=True)

    role_vectors_scaled = scaler_obj.transform(vectors_at_layer.numpy())
    role_vectors_norm = role_vectors_scaled / np.linalg.norm(role_vectors_scaled, axis=1, keepdims=True)

    role_cosine_sims = role_vectors_norm @ pc_directions.T

    default_at_layer = default_vector[s, TARGET_LAYER].float().numpy().reshape(1, -1)
    default_scaled = scaler_obj.transform(default_at_layer)
    default_norm = default_scaled / np.linalg.norm(default_scaled)
    default_cosine_sims = (default_norm @ pc_directions.T)[0]

    fig = plot_pc_lines(
        role_cosine_sims, role_labels,
        default_cosine_sims=default_cosine_sims,
        figsize=(8, 7), n_extremes=20, show_histogram=True, n_pcs=n_pcs,
    )
    slot_label = display_label(result['label'])
    plt.suptitle(f"PC Cosine Similarity — {slot_label} ({model_title})", y=1.01)
    plt.show()


In [ ]:
for result in pca_results:
    s = result["slot_idx"]
    pca_obj = result["pca"]
    scaler_obj = result["scaler"]
    vectors_at_layer = result["vectors_at_layer"]
    slot_label = display_label(result['label'])

    pc_directions = pca_obj.components_[:N_PCS]
    pc_directions = pc_directions / np.linalg.norm(pc_directions, axis=1, keepdims=True)

    role_vectors_scaled = scaler_obj.transform(vectors_at_layer.numpy())
    role_vectors_norm = role_vectors_scaled / np.linalg.norm(role_vectors_scaled, axis=1, keepdims=True)
    role_cosine_sims = role_vectors_norm @ pc_directions.T

    print(f"\n{'='*80}")
    print(f"  {slot_label} ({model_title})")
    print(f"{'='*80}")

    for pc_idx in range(min(N_PCS, role_cosine_sims.shape[1])):
        scores = role_cosine_sims[:, pc_idx]
        ranked = sorted(zip(role_labels, scores), key=lambda x: x[1])
        print(f"\n--- PC{pc_idx+1} (negative → positive) ---")
        for rank, (name, score) in enumerate(ranked):
            print(f"  {rank+1:3d}. {score:+.4f}  {name}")